In [11]:
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [13]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [15]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [17]:
print(uploader.value)

({'name': 'fridge_items.png', 'type': 'image/png', 'size': 1455032, 'content': <memory at 0x10ba0d780>, 'last_modified': datetime.datetime(2026, 9, 5, 21, 20, 34, 48000, tzinfo=datetime.timezone.utc)},)


In [18]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [19]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [20]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "I have some leftover items in fridge. What can I make?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}, 
    config
)

print(response['messages'][-1].content)

Nice—looks like you’ve got some bread, milk, and a protein powder on hand. Here are quick ideas that mostly use those items (plus a few optional add-ins you might have):

Quick ideas you can make now
- Protein smoothie or shake
  - Milk + protein powder + ice (and a banana or berries if you have them)
  - Quick, filling breakfast or post-workout drink

- Creamy bread pudding cups (no oven needed if you have a microwave)
  - Tear bread into chunks
  - Whisk milk + a scoop of protein powder (optional a little sugar or cocoa)
  - Soak the bread, then microwave until set (about 2–4 minutes). Chill or serve warm.

- Protein mug “cake” or pancake
  - In a mug: mix protein powder + a splash of milk + a bit of flour or oats if you have them
  - Microwave until firm (1–2 minutes). Eat as a quick breakfast or snack.

- Milk-soaked French toast (needs eggs)
  - If you have eggs: whisk eggs with milk, dip bread slices, and cook on a pan with a little butter. Top with a pinch of cinnamon or sugar.


In [21]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'I have some leftover items in fridge. What can I make?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='e770cb87-15de-46ab-ae45-dd59d177c141'),
              AIMessage(content='Nice—looks like you’ve got some bread, milk, and a protein powder on hand. Here are quick ideas that mostly use those items (plus a few optional add-ins you might have):\n\nQuick ideas you can make now\n- Protein smoothie or shake\n  - Milk + protein powder + ice (and a banana or berries if you have them)\n  - Quick, filling breakfast or post-workout drink\n\n- Creamy bread pudding cups (no oven needed if you have a microwave)\n  - Tear bread into chunks\n  - Whisk milk + a scoop of protein powder (optional a little sugar or cocoa)\n  - Soak the bread, then microwave until set (about 2–4 minutes). Chill or serve warm.\n\n- Protein mug “cake” or pancake\n  - In a mug: mix protein powder +